先造数据
  ↓
再把主单 + 明细变成模型特征
  ↓
再训练 One-Class SVM
  ↓
再给每条记录打风险分
  ↓
再评估模型抓得怎么样
  ↓
最后生成审核人员能看懂的异常原因

就诊主单级异常检测模型
│
├── 0. 导入依赖库
│
├── 1. 生成模拟数据
│   ├── 1.1 定义基础参数
│   ├── 1.2 定义诊断画像 diag_profiles
│   ├── 1.3 定义费用类别 fee_categories
│   ├── 1.4 定义 add_visit() 函数
│   ├── 1.5 生成正常就诊数据
│   ├── 1.6 注入异常1：同病种高额费用 high_cost
│   ├── 1.7 注入异常2：药品费用占比异常 high_drug_ratio
│   ├── 1.8 注入异常3：短期高频就诊 frequent_visit
│   └── 1.9 注入异常4：疑似拆分结算 split_bill
│
├── 2. 构建模型特征表
│   ├── 2.1 费用明细透视表 fee_pivot
│   ├── 2.2 保证 DRUG / EXAM / TREAT / MATERIAL 四类字段存在
│   ├── 2.3 聚合明细统计特征 fee_agg
│   ├── 2.4 主单表合并费用类别特征
│   ├── 2.5 主单表合并明细统计特征
│   ├── 2.6 生成费用结构比例特征
│   ├── 2.7 生成同诊断费用倍率 diag_cost_ratio
│   └── 2.8 生成时间窗口特征 add_time_features()
│
├── 3. 训练 One-Class SVM 模型
│   ├── 3.1 选择数值特征 num_cols
│   ├── 3.2 选择类别特征 cat_cols
│   ├── 3.3 构建输入数据 X
│   ├── 3.4 数据预处理 ColumnTransformer
│   │   ├── 数值字段：StandardScaler
│   │   └── 类别字段：OneHotEncoder
│   ├── 3.5 定义 OneClassSVM 模型
│   └── 3.6 构建 Pipeline 并训练
│
├── 4. 模型预测与风险分生成
│   ├── 4.1 predict() 输出正常 / 异常标签
│   ├── 4.2 生成 is_svm_anomaly 字段
│   ├── 4.3 decision_function() 输出原始分数
│   └── 4.4 转换成 0-100 风险分 risk_score
│
├── 5. 模型效果评估
│   ├── 5.1 生成交叉表 cross_tab
│   ├── 5.2 计算 Top K 命中率
│   └── 5.3 分析模型最容易识别哪类异常
│
├── 6. 生成异常原因说明
│   ├── 6.1 定义 build_reason() 函数
│   ├── 6.2 根据阈值生成中文原因
│   └── 6.3 生成 reason_summary 字段
│
└── 7. 导出审核线索清单
    ├── 7.1 选择输出字段
    ├── 7.2 按 risk_score 排序
    └── 7.3 导出 CSV 文件

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

np.random.seed(42)

# -----------------------------
# 1. 定义基础参数
# -----------------------------
n_normal = 2500

diag_profiles = {
    "J00_感冒":       {"mean": 180,  "ratios": [0.55, 0.15, 0.25, 0.05]},
    "I10_高血压":     {"mean": 260,  "ratios": [0.70, 0.08, 0.17, 0.05]},
    "E11_糖尿病":     {"mean": 320,  "ratios": [0.68, 0.10, 0.17, 0.05]},
    "M54_腰痛":       {"mean": 420,  "ratios": [0.25, 0.35, 0.30, 0.10]},
    "K29_胃炎":       {"mean": 240,  "ratios": [0.60, 0.18, 0.17, 0.05]},
    "N39_尿路感染":   {"mean": 280,  "ratios": [0.58, 0.22, 0.15, 0.05]},
}

fee_categories = ["DRUG", "EXAM", "TREAT", "MATERIAL"]
start_date = datetime(2025, 1, 1)

visit_rows = []
detail_rows = []
visit_id_counter = 1


def add_visit(patient_id, org_id, doctor_id, visit_date, diag_code, total_amount,
              ratios, true_label=0, anomaly_type="normal"):
    """
    添加一条就诊主单 + 多条费用明细
    true_label: 0=正常，1=我们人为埋入的异常
    """
    global visit_id_counter, visit_rows, detail_rows
    
    visit_id = f"V{visit_id_counter:06d}"
    visit_id_counter += 1
    
    fund_amount = total_amount * np.random.uniform(0.55, 0.85)
    self_amount = total_amount - fund_amount
    
    visit_rows.append({
        "pk_visit": visit_id,
        "pk_patient": f"P{patient_id:04d}",
        "pk_org": f"ORG{org_id:03d}",
        "pk_doctor": f"D{doctor_id:03d}",
        "visit_date": visit_date,
        "diag_code": diag_code,
        "total_amount": round(total_amount, 2),
        "fund_amount": round(fund_amount, 2),
        "self_amount": round(self_amount, 2),
        "true_label": true_label,
        "anomaly_type": anomaly_type
    })
    
    # 生成费用明细：每类费用拆成若干项目
    for cat, ratio in zip(fee_categories, ratios):
        cat_amount = total_amount * ratio
        
        if cat_amount <= 1:
            continue
        
        n_items = np.random.randint(1, 4)
        parts = np.random.dirichlet(np.ones(n_items)) * cat_amount
        
        for i, amt in enumerate(parts):
            quantity = np.random.randint(1, 5)
            unit_price = round(amt / quantity, 2)
            
            detail_rows.append({
                "pk_visit": visit_id,
                "fee_category": cat,
                "item_code": f"{cat}_{np.random.randint(1, 100):03d}",
                "item_name": f"{cat}_项目_{i+1}",
                "quantity": quantity,
                "unit_price": unit_price,
                "amount": round(amt, 2)
            })


# -----------------------------
# 2. 生成正常就诊
# -----------------------------
for _ in range(n_normal):
    patient_id = np.random.randint(1, 600)
    org_id = np.random.randint(1, 30)
    doctor_id = np.random.randint(1, 100)
    visit_date = start_date + timedelta(days=int(np.random.randint(0, 180)))
    
    diag_code = np.random.choice(list(diag_profiles.keys()))
    profile = diag_profiles[diag_code]
    
    # 用对数正态分布模拟费用：大多数正常，少数自然偏高
    total_amount = np.random.lognormal(mean=np.log(profile["mean"]), sigma=0.35)
    
    # 费用结构比例
    base_ratios = np.array(profile["ratios"])
    ratios = np.random.dirichlet(base_ratios * 30)
    
    add_visit(
        patient_id=patient_id,
        org_id=org_id,
        doctor_id=doctor_id,
        visit_date=visit_date,
        diag_code=diag_code,
        total_amount=total_amount,
        ratios=ratios,
        true_label=0,
        anomaly_type="normal"
    )


# -----------------------------
# 3. 注入异常1：同病种高额费用异常
# -----------------------------
for _ in range(35):
    patient_id = np.random.randint(1, 600)
    org_id = np.random.randint(1, 30)
    doctor_id = np.random.randint(1, 100)
    visit_date = start_date + timedelta(days=int(np.random.randint(0, 180)))
    
    diag_code = np.random.choice(list(diag_profiles.keys()))
    profile = diag_profiles[diag_code]
    
    total_amount = profile["mean"] * np.random.uniform(4.0, 8.0)
    ratios = np.random.dirichlet(np.array(profile["ratios"]) * 20)
    
    add_visit(patient_id, org_id, doctor_id, visit_date, diag_code,
              total_amount, ratios, 1, "high_cost")


# -----------------------------
# 4. 注入异常2：药品费用占比异常高
# -----------------------------
for _ in range(35):
    patient_id = np.random.randint(1, 600)
    org_id = np.random.randint(1, 30)
    doctor_id = np.random.randint(1, 100)
    visit_date = start_date + timedelta(days=int(np.random.randint(0, 180)))
    
    diag_code = np.random.choice(list(diag_profiles.keys()))
    profile = diag_profiles[diag_code]
    
    total_amount = profile["mean"] * np.random.uniform(1.2, 2.0)
    ratios = [0.92, 0.03, 0.03, 0.02]
    
    add_visit(patient_id, org_id, doctor_id, visit_date, diag_code,
              total_amount, ratios, 1, "high_drug_ratio")


# -----------------------------
# 5. 注入异常3：短期高频就诊
# -----------------------------
for patient_id in np.random.choice(range(1, 600), size=20, replace=False):
    base_day = int(np.random.randint(0, 170))
    diag_code = np.random.choice(list(diag_profiles.keys()))
    profile = diag_profiles[diag_code]
    
    for j in range(5):
        org_id = np.random.randint(1, 30)
        doctor_id = np.random.randint(1, 100)
        visit_date = start_date + timedelta(days=base_day + j)
        total_amount = profile["mean"] * np.random.uniform(0.8, 1.5)
        ratios = np.random.dirichlet(np.array(profile["ratios"]) * 30)
        
        add_visit(patient_id, org_id, doctor_id, visit_date, diag_code,
                  total_amount, ratios, 1, "frequent_visit")


# -----------------------------
# 6. 注入异常4：疑似拆分结算
# -----------------------------
for patient_id in np.random.choice(range(1, 600), size=15, replace=False):
    visit_date = start_date + timedelta(days=int(np.random.randint(0, 180)))
    org_id = np.random.randint(1, 30)
    doctor_id = np.random.randint(1, 100)
    diag_code = np.random.choice(list(diag_profiles.keys()))
    
    for j in range(3):
        total_amount = np.random.uniform(850, 990)  # 故意接近某个假设审核阈值
        ratios = [0.45, 0.25, 0.25, 0.05]
        
        add_visit(patient_id, org_id, doctor_id, visit_date, diag_code,
                  total_amount, ratios, 1, "split_bill")


visit_header = pd.DataFrame(visit_rows)
fee_detail = pd.DataFrame(detail_rows)

print("就诊主单 visit_header：", visit_header.shape)
print("费用明细 fee_detail：", fee_detail.shape)

display(visit_header.head())
display(fee_detail.head())

就诊主单 visit_header： (2715, 11)
费用明细 fee_detail： (21373, 7)


,pk_visit,pk_patient,pk_org,pk_doctor,visit_date,diag_code,total_amount,fund_amount,self_amount,true_label,anomaly_type
0,V000001,P0103,ORG020,D093,2025-01-15,E11_糖尿病,377.73,314.11,63.62,0,normal
1,V000002,P0340,ORG028,D060,2025-06-21,K29_胃炎,254.19,142.19,112.00,0,normal
2,V000003,P0344,ORG001,D008,2025-03-04,E11_糖尿病,399.83,264.38,135.44,0,normal
3,V000004,P0485,ORG023,D015,2025-06-20,K29_胃炎,360.79,295.05,65.75,0,normal
4,V000005,P0564,ORG012,D039,2025-05-10,E11_糖尿病,219.85,153.20,66.65,0,normal


,pk_visit,fee_category,item_code,item_name,quantity,unit_price,amount
0,V000001,DRUG,DRUG_058,DRUG_项目_1,4,38.12,152.47
1,V000001,DRUG,DRUG_089,DRUG_项目_2,2,76.97,153.94
2,V000001,EXAM,EXAM_042,EXAM_项目_1,3,9.57,28.71
3,V000001,TREAT,TREAT_064,TREAT_项目_1,3,2.95,8.84
4,V000001,TREAT,TREAT_003,TREAT_项目_2,1,22.32,22.32


In [ ]:
# -----------------------------
# 1. 费用明细聚合
# -----------------------------
fee_pivot = fee_detail.pivot_table(
    index="pk_visit",
    columns="fee_category",
    values="amount",
    aggfunc="sum",
    fill_value=0
).reset_index()

fee_pivot.columns.name = None

# 保证四类字段都存在
for col in fee_categories:
    if col not in fee_pivot.columns:
        fee_pivot[col] = 0

fee_agg = fee_detail.groupby("pk_visit").agg(
    detail_total_amount=("amount", "sum"),
    item_count=("item_code", "count"),
    distinct_item_count=("item_code", "nunique"),
    max_item_amount=("amount", "max")
).reset_index()

feature_df = visit_header.merge(fee_pivot, on="pk_visit", how="left")
feature_df = feature_df.merge(fee_agg, on="pk_visit", how="left")

# -----------------------------
# 2. 费用结构比例特征
# -----------------------------
feature_df["drug_amount"] = feature_df["DRUG"]
feature_df["exam_amount"] = feature_df["EXAM"]
feature_df["treat_amount"] = feature_df["TREAT"]
feature_df["material_amount"] = feature_df["MATERIAL"]

feature_df["drug_ratio"] = feature_df["drug_amount"] / feature_df["total_amount"]
feature_df["exam_ratio"] = feature_df["exam_amount"] / feature_df["total_amount"]
feature_df["treat_ratio"] = feature_df["treat_amount"] / feature_df["total_amount"]
feature_df["material_ratio"] = feature_df["material_amount"] / feature_df["total_amount"]
feature_df["max_item_ratio"] = feature_df["max_item_amount"] / feature_df["total_amount"]

# -----------------------------
# 3. 同诊断费用倍率
# -----------------------------
diag_avg = feature_df.groupby("diag_code")["total_amount"].mean().rename("diag_avg_amount")
feature_df = feature_df.merge(diag_avg, on="diag_code", how="left")
feature_df["diag_cost_ratio"] = feature_df["total_amount"] / feature_df["diag_avg_amount"]

# -----------------------------
# 4. 时间频次特征
# -----------------------------
feature_df["visit_date"] = pd.to_datetime(feature_df["visit_date"])
feature_df = feature_df.sort_values(["pk_patient", "visit_date"])

def add_time_features(group):
    dates = group["visit_date"].values
    orgs = group["pk_org"].values
    
    cnt_7d = []
    org_cnt_30d = []
    
    for i, d in enumerate(dates):
        start_7d = d - np.timedelta64(7, "D")
        start_30d = d - np.timedelta64(30, "D")
        
        mask_7d = (dates >= start_7d) & (dates <= d)
        mask_30d = (dates >= start_30d) & (dates <= d)
        
        cnt_7d.append(mask_7d.sum())
        org_cnt_30d.append(len(set(orgs[mask_30d])))
    
    group["patient_visit_cnt_7d"] = cnt_7d
    group["patient_org_cnt_30d"] = org_cnt_30d
    return group

feature_df = feature_df.groupby("pk_patient", group_keys=False).apply(add_time_features)

print("模型特征表：", feature_df.shape)

feature_df.head()
display(feature_df[[
    "pk_visit", "pk_patient", "pk_org", "pk_doctor", "visit_date", "diag_code",
    "total_amount", "drug_ratio", "exam_ratio", "item_count",
    "patient_visit_cnt_7d", "patient_org_cnt_30d", "diag_cost_ratio",
    "true_label", "anomaly_type"
]].head(10))

模型特征表： (2715, 32)


C:\Users\hhsjo\AppData\Local\Temp\ipykernel_27228\645694982.py:77: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  feature_df = feature_df.groupby("pk_patient", group_keys=False).apply(add_time_features)


,pk_visit,pk_patient,pk_org,pk_doctor,visit_date,diag_code,total_amount,fund_amount,self_amount,true_label,...,material_amount,drug_ratio,exam_ratio,treat_ratio,material_ratio,max_item_ratio,diag_avg_amount,diag_cost_ratio,patient_visit_cnt_7d,patient_org_cnt_30d
1383,V001384,P0001,ORG010,D059,2025-02-04,N39_尿路感染,284.81,219.85,64.96,0,...,2.94,0.714301,0.204171,0.071205,0.010323,0.470033,318.085390,0.895388,1,1
988,V000989,P0001,ORG023,D031,2025-02-07,N39_尿路感染,467.11,325.69,141.42,0,...,10.41,0.668194,0.170602,0.138897,0.022286,0.624157,318.085390,1.468505,2,2
1267,V001268,P0001,ORG017,D018,2025-06-09,J00_感冒,135.24,107.84,27.41,0,...,12.96,0.447057,0.112689,0.344499,0.095830,0.272331,227.962556,0.593255,1,1
2438,V002439,P0001,ORG013,D091,2025-06-27,E11_糖尿病,524.66,353.35,171.31,0,...,28.61,0.770156,0.107327,0.068025,0.054531,0.770156,386.231784,1.358407,1,2
2567,V002568,P0002,ORG006,D091,2025-01-08,N39_尿路感染,381.04,210.82,170.22,1,...,7.62,0.919982,0.029997,0.029997,0.019998,0.866838,318.085390,1.197917,1,1


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import OneClassSVM

# -----------------------------
# 1. 选择模型输入特征
# -----------------------------
num_cols = [
    "total_amount",
    "fund_amount",
    "self_amount",
    "drug_amount",
    "exam_amount",
    "treat_amount",
    "material_amount",
    "drug_ratio",
    "exam_ratio",
    "treat_ratio",
    "material_ratio",
    "item_count",
    "distinct_item_count",
    "max_item_amount",
    "max_item_ratio",
    "diag_cost_ratio",
    "patient_visit_cnt_7d",
    "patient_org_cnt_30d"
]

cat_cols = [
    "diag_code",
    "pk_org"
]

X = feature_df[num_cols + cat_cols].copy()

# -----------------------------
# 2. 数据预处理
# 数值字段标准化，类别字段独热编码
# -----------------------------
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]
)

# -----------------------------
# 3. One-Class SVM
# nu 大致表示希望模型识别出的异常比例
# 这里先设 5%
# -----------------------------
model = OneClassSVM(
    kernel="rbf",
    nu=0.05,
    gamma="scale"
)

pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", model)
])

pipeline.fit(X)

# -----------------------------
# 4. 预测结果
# predict: 1=正常，-1=异常
# decision_function: 越低越异常
# -----------------------------
feature_df["svm_pred"] = pipeline.predict(X)
feature_df["is_svm_anomaly"] = (feature_df["svm_pred"] == -1).astype(int)

raw_score = pipeline.decision_function(X)
feature_df["raw_score"] = raw_score

# 转成 0-100 风险分，越高越异常
scaler = MinMaxScaler(feature_range=(0, 100))
feature_df["risk_score"] = scaler.fit_transform((-raw_score).reshape(-1, 1))

print("模型识别出的异常数量：", feature_df["is_svm_anomaly"].sum())
print("总样本数：", len(feature_df))
print("异常比例：", round(feature_df["is_svm_anomaly"].mean() * 100, 2), "%")

display(feature_df[[
    "pk_visit", "pk_patient", "diag_code", "total_amount",
    "drug_ratio", "patient_visit_cnt_7d", "diag_cost_ratio",
    "risk_score", "is_svm_anomaly", "true_label", "anomaly_type"
]].sort_values("risk_score", ascending=False).head(20))

模型识别出的异常数量： 137
总样本数： 2715
异常比例： 5.05 %


,pk_visit,pk_patient,diag_code,total_amount,drug_ratio,patient_visit_cnt_7d,diag_cost_ratio,risk_score,is_svm_anomaly,true_label,anomaly_type
2532,V002533,P0150,M54_腰痛,3005.28,0.189160,1,6.067990,100.000000,1,1,high_cost
2509,V002510,P0349,E11_糖尿病,2316.51,0.714489,2,5.997720,99.710947,1,1,high_cost
2534,V002535,P0542,M54_腰痛,3157.73,0.164587,2,6.375803,99.693630,1,1,high_cost
2527,V002528,P0110,M54_腰痛,2200.25,0.131317,3,4.442546,99.498031,1,1,high_cost
2521,V002522,P0292,M54_腰痛,2247.46,0.242732,1,4.537868,99.361636,1,1,high_cost
2516,V002517,P0407,M54_腰痛,3130.22,0.324504,1,6.320257,98.835917,1,1,high_cost
2523,V002524,P0310,E11_糖尿病,2363.88,0.598994,1,6.120366,98.590719,1,1,high_cost
2504,V002505,P0225,M54_腰痛,2705.45,0.233166,1,5.462600,98.543094,1,1,high_cost
2531,V002532,P0069,N39_尿路感染,2006.67,0.481110,3,6.308589,98.178591,1,1,high_cost
2528,V002529,P0143,E11_糖尿病,2265.58,0.576709,2,5.865856,98.022779,1,1,high_cost


In [ ]:
# -----------------------------
# 1. 整体交叉表
# -----------------------------
cross_tab = pd.crosstab(
    feature_df["true_label"],
    feature_df["is_svm_anomaly"],
    rownames=["是否真实异常"],
    colnames=["SVM是否识别异常"]
)

display(cross_tab)

# -----------------------------
# 2. 不同异常类型的识别情况
# -----------------------------
# 看
type_result = feature_df.groupby("anomaly_type").agg(
    total_count=("pk_visit", "count"),
    detected_count=("is_svm_anomaly", "sum"),
    avg_risk_score=("risk_score", "mean")
)

type_result["detected_rate"] = type_result["detected_count"] / type_result["total_count"]

display(type_result.sort_values("detected_rate", ascending=False))

# -----------------------------
# 3. 指标
# -----------------------------
TN = 2426
FP = 74
FN = 151
TP = 64

precision = TP / (TP + FP)
recall = TP / (TP + FN)
specificity = TN / (TN + FP)
accuracy = (TP + TN) / (TP + TN + FP + FN)

print("精确率 Precision：", round(precision * 100, 2), "%")
print("召回率 Recall：", round(recall * 100, 2), "%")
print("正常识别率 Specificity：", round(specificity * 100, 2), "%")
print("准确率 Accuracy：", round(accuracy * 100, 2), "%")

# -----------------------------
# 4. Top K 命中率
# -----------------------------
for k in [20, 50, 100, 200]:
    topk = feature_df.sort_values("risk_score", ascending=False).head(k)
    precision_at_k = topk["true_label"].mean()
    print(f"Top {k} 中真实异常比例：{precision_at_k:.2%}")

# -----------------------------
# 5. 看模型最容易抓到哪类异常
# -----------------------------
top100 = feature_df.sort_values("risk_score", ascending=False).head(100)

display(
    top100.groupby("anomaly_type").agg(
        count=("pk_visit", "count"),
        avg_risk_score=("risk_score", "mean"),
        avg_total_amount=("total_amount", "mean"),
        avg_drug_ratio=("drug_ratio", "mean"),
        avg_visit_cnt_7d=("patient_visit_cnt_7d", "mean")
    ).sort_values("count", ascending=False)
)

SVM是否识别异常,0,1
是否真实异常,,
0,2431,69
1,147,68


,total_count,detected_count,avg_risk_score,detected_rate
anomaly_type,,,,
high_cost,35,35,89.049637,1.000000
high_drug_ratio,35,11,44.305030,0.314286
split_bill,45,7,42.615076,0.155556
frequent_visit,100,15,33.679908,0.150000
normal,2500,69,27.468215,0.027600


精确率 Precision： 46.38 %
召回率 Recall： 29.77 %
正常识别率 Specificity： 97.04 %
准确率 Accuracy： 91.71 %
Top 20 中真实异常比例：100.00%
Top 50 中真实异常比例：74.00%
Top 100 中真实异常比例：59.00%
Top 200 中真实异常比例：41.50%


,count,avg_risk_score,avg_total_amount,avg_drug_ratio,avg_visit_cnt_7d
anomaly_type,,,,,
normal,41,55.977564,459.362927,0.386787,1.585366
high_cost,35,89.049637,1869.742571,0.527475,1.257143
frequent_visit,12,54.319746,378.198333,0.448157,5.250000
high_drug_ratio,7,54.587318,549.345714,0.919997,1.428571
split_bill,5,55.757465,912.146000,0.450000,3.400000


In [ ]:
total_amount_p98 = feature_df["total_amount"].quantile(0.98)

def build_reason(row):
    reasons = []
    
    if row["diag_cost_ratio"] >= 2.5:
        reasons.append(f"同诊断费用倍率较高：{row['diag_cost_ratio']:.2f}倍")
    
    if row["drug_ratio"] >= 0.85:
        reasons.append(f"药品费用占比偏高：{row['drug_ratio']:.1%}")
    
    if row["exam_ratio"] >= 0.60:
        reasons.append(f"检查费用占比偏高：{row['exam_ratio']:.1%}")
    
    if row["patient_visit_cnt_7d"] >= 4:
        reasons.append(f"患者近7天就诊次数较多：{int(row['patient_visit_cnt_7d'])}次")
    
    if row["patient_org_cnt_30d"] >= 4:
        reasons.append(f"患者近30天就诊机构数较多：{int(row['patient_org_cnt_30d'])}家")
    
    if row["max_item_ratio"] >= 0.70:
        reasons.append(f"最大单项费用占比较高：{row['max_item_ratio']:.1%}")
    
    if row["total_amount"] >= total_amount_p98:
        reasons.append(f"总费用处于全量样本前2%：{row['total_amount']:.2f}元")
    
    if len(reasons) == 0:
        reasons.append("综合费用结构、频次和同病种对比结果偏离正常模式")
    
    return "；".join(reasons)

feature_df["reason_summary"] = feature_df.apply(build_reason, axis=1)

suspicious_list = feature_df.sort_values("risk_score", ascending=False).head(50)

display(suspicious_list[[
    "pk_visit",
    "pk_patient",
    "pk_org",
    "pk_doctor",
    "visit_date",
    "diag_code",
    "total_amount",
    "drug_ratio",
    "patient_visit_cnt_7d",
    "diag_cost_ratio",
    "risk_score",
    "is_svm_anomaly",
    "reason_summary",
    "true_label",
    "anomaly_type"
]])

,pk_visit,pk_patient,pk_org,pk_doctor,visit_date,diag_code,total_amount,drug_ratio,patient_visit_cnt_7d,diag_cost_ratio,risk_score,is_svm_anomaly,reason_summary,true_label,anomaly_type
2532,V002533,P0150,ORG012,D060,2025-02-27,M54_腰痛,3005.28,0.189160,1,6.067990,100.000000,1,同诊断费用倍率较高：6.07倍；总费用处于全量样本前2%：3005.28元,1,high_cost
2509,V002510,P0349,ORG018,D070,2025-05-14,E11_糖尿病,2316.51,0.714489,2,5.997720,99.710947,1,同诊断费用倍率较高：6.00倍；最大单项费用占比较高：71.4%；总费用处于全量样本前2%：...,1,high_cost
2534,V002535,P0542,ORG005,D016,2025-02-16,M54_腰痛,3157.73,0.164587,2,6.375803,99.693630,1,同诊断费用倍率较高：6.38倍；总费用处于全量样本前2%：3157.73元,1,high_cost
2527,V002528,P0110,ORG026,D094,2025-05-24,M54_腰痛,2200.25,0.131317,3,4.442546,99.498031,1,同诊断费用倍率较高：4.44倍；总费用处于全量样本前2%：2200.25元,1,high_cost
2521,V002522,P0292,ORG019,D048,2025-06-19,M54_腰痛,2247.46,0.242732,1,4.537868,99.361636,1,同诊断费用倍率较高：4.54倍；总费用处于全量样本前2%：2247.46元,1,high_cost
2516,V002517,P0407,ORG007,D007,2025-01-21,M54_腰痛,3130.22,0.324504,1,6.320257,98.835917,1,同诊断费用倍率较高：6.32倍；总费用处于全量样本前2%：3130.22元,1,high_cost
2523,V002524,P0310,ORG025,D073,2025-03-21,E11_糖尿病,2363.88,0.598994,1,6.120366,98.590719,1,同诊断费用倍率较高：6.12倍；总费用处于全量样本前2%：2363.88元,1,high_cost
2504,V002505,P0225,ORG025,D082,2025-01-29,M54_腰痛,2705.45,0.233166,1,5.462600,98.543094,1,同诊断费用倍率较高：5.46倍；总费用处于全量样本前2%：2705.45元,1,high_cost
2531,V002532,P0069,ORG011,D099,2025-05-20,N39_尿路感染,2006.67,0.481110,3,6.308589,98.178591,1,同诊断费用倍率较高：6.31倍；总费用处于全量样本前2%：2006.67元,1,high_cost
2528,V002529,P0143,ORG002,D027,2025-04-21,E11_糖尿病,2265.58,0.576709,2,5.865856,98.022779,1,同诊断费用倍率较高：5.87倍；总费用处于全量样本前2%：2265.58元,1,high_cost


In [ ]:
output_cols = [
    "pk_visit",
    "pk_patient",
    "pk_org",
    "pk_doctor",
    "visit_date",
    "diag_code",
    "total_amount",
    "fund_amount",
    "self_amount",
    "drug_amount",
    "exam_amount",
    "treat_amount",
    "material_amount",
    "drug_ratio",
    "exam_ratio",
    "patient_visit_cnt_7d",
    "patient_org_cnt_30d",
    "diag_cost_ratio",
    "risk_score",
    "is_svm_anomaly",
    "reason_summary",
    "true_label",
    "anomaly_type"
]

result_df = feature_df.sort_values("risk_score", ascending=False)[output_cols]

result_df.to_csv("svm_visit_anomaly_result.csv", index=False, encoding="utf-8-sig")

print("已导出：svm_visit_anomaly_result.csv")

已导出：svm_visit_anomaly_result.csv
